<a href="https://colab.research.google.com/github/24dhanush/GUVI_PROJECT/blob/https%2Fgithub.com%2F24dhanush%2FGUVI_PROJECT%2Ftree%2Fmain%2Fproject%25203/fitapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install streamlit


In [ ]:
%%writefile fitapp.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from xgboost import XGBRegressor

# -----------------------------
# Page Configuration
# -----------------------------
st.set_page_config(
    page_title="Fitness Prediction System",
    page_icon="🏋",
    layout="wide"
)

st.title("🏋 Fitness Prediction System")

st.subheader("3,Calorie Burn Prediction & Workout Pattern Clustering Using Fitbit Data ")
st.write("Calorie Burn Prediction & Workout Patterns")

st.subheader("About Me")
st.write("Dhanush DB")


# -----------------------------
# Sidebar
# -----------------------------
menu = st.sidebar.selectbox(
    "Select Module",
    ("Supervised Learning", "Unsupervised Learning")
)

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("/content/Fitbit_dataset.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns="Unnamed: 0")

st.subheader("Original Dataset")
st.dataframe(df)

# -----------------------------
# Data Preprocessing
# -----------------------------
encoder = LabelEncoder()

df["Gender"] = encoder.fit_transform(df["Gender"])
df["Workout_Type"] = encoder.fit_transform(df["Workout_Type"])

scaler = StandardScaler()

scaled = scaler.fit_transform(df)

final_data = pd.DataFrame(
    scaled,
    columns=df.columns
)

st.subheader("Preprocessed Dataset")
st.dataframe(final_data)

# ===========================================================
# SUPERVISED
# ===========================================================
if menu == "Supervised Learning":

    st.header("Supervised Learning Models")

    X = final_data.drop("Calories_Burned (kcal)", axis=1)
    y = final_data["Calories_Burned (kcal)"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=3
    )

    models = [
        LinearRegression(),
        Ridge(),
        SVR(),
        RandomForestRegressor(random_state=42),
        DecisionTreeRegressor(random_state=42),
        Lasso(),
        XGBRegressor(random_state=42)
    ]

    results = []

    for model in models:

        model.fit(X_train, y_train)

        prediction = model.predict(X_test)

        mae = mean_absolute_error(y_test, prediction)
        rmse = np.sqrt(mean_squared_error(y_test, prediction))
        r2 = r2_score(y_test, prediction)

        results.append({
            "Model": model.__class__.__name__,
            "MAE": round(mae,4),
            "RMSE": round(rmse,4),
            "R2 Score": round(r2,4)
        })

    results_df = pd.DataFrame(results)

    st.subheader("Model Comparison")

    st.dataframe(results_df, use_container_width=True)

    best_model = results_df.loc[
        results_df["R2 Score"].idxmax()
    ]

    st.success(
        f"Best Model : {best_model['Model']}   |   R² Score = {best_model['R2 Score']}"
    )

    fig, ax = plt.subplots(1,3,figsize=(18,5))

    sns.barplot(
        x="Model",
        y="MAE",
        data=results_df,
        ax=ax[0]
    )

    ax[0].tick_params(axis='x', rotation=45)
    ax[0].set_title("MAE")

    sns.barplot(
        x="Model",
        y="RMSE",
        data=results_df,
        ax=ax[1]
    )

    ax[1].tick_params(axis='x', rotation=45)
    ax[1].set_title("RMSE")

    sns.barplot(
        x="Model",
        y="R2 Score",
        data=results_df,
        ax=ax[2]
    )

    ax[2].tick_params(axis='x', rotation=45)
    ax[2].set_title("R² Score")

    st.pyplot(fig)

# ===========================================================
# UNSUPERVISED
# ===========================================================
else:

    st.header("Unsupervised Learning")

    cluster_data = final_data.drop(
        columns=["Workout_Type", "Calories_Burned (kcal)"]
    )

    pca = PCA(
        n_components=0.95,
        random_state=42
    )

    principal_components = pca.fit_transform(cluster_data)

    # --------------------------
    # Elbow Method
    # --------------------------
    wcss = []

    for i in range(2,11):

        kmeans = KMeans(
            n_clusters=i,
            init="k-means++",
            random_state=42,
            n_init=10
        )

        kmeans.fit(principal_components)

        wcss.append(kmeans.inertia_)

    st.subheader("Elbow Method")

    fig1, ax1 = plt.subplots(figsize=(7,5))

    ax1.plot(range(2,11), wcss)

    ax1.set_xlabel("Number of Clusters")

    ax1.set_ylabel("WCSS")

    st.pyplot(fig1)

    # --------------------------
    # KMeans
    # --------------------------
    kmeans = KMeans(
        n_clusters=3,
        random_state=42,
        n_init=10
    )

    Y = kmeans.fit_predict(principal_components)

    st.subheader("Cluster Visualization")

    fig2, ax2 = plt.subplots(figsize=(8,6))

    sns.scatterplot(
        x=principal_components[:,0],
        y=principal_components[:,1],
        hue=Y,
        palette="viridis",
        s=60,
        ax=ax2
    )

    ax2.scatter(
        kmeans.cluster_centers_[:,0],
        kmeans.cluster_centers_[:,1],
        marker="X",
        s=200,
        color="red"
    )

    ax2.set_title("Workout Pattern Clusters")

    st.pyplot(fig2)

    score = silhouette_score(principal_components, Y)

    st.metric(
        "Silhouette Score",
        round(score,3)
    )

    if score >= 0.15:
        st.success("Cluster Accepted")
    else:
        st.error("Cluster Not Accepted")

    final_data["Cluster"] = Y

    cluster_summary = final_data.groupby("Cluster").mean(numeric_only=True)

    st.subheader("Cluster Summary")

    st.dataframe(cluster_summary)
    st.subheader("Heatmap")

    fig3, ax3 = plt.subplots(figsize=(12,5))

    sns.heatmap(
        cluster_summary,
        annot=True,
        cmap="coolwarm",
        fmt=".2f",
        ax=ax3
    )

    st.pyplot(fig3)

# -----------------------------
# Footer
# -----------------------------

st.sidebar.markdown("---")
st.sidebar.write("Created by Dhanush DB")

In [ ]:
!streamlit run fitapp.py & npx localtunnel --port 8501